<a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/252-tabular-experiment/weightslab/examples/Notebooks/PyTorch/ws-fraud-detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">

  <a href="https://grayboxtech.github.io/weightslab/latest/index.html" target="_blank">
    <img width="100%" src="https://raw.githubusercontent.com/GrayboxTech/.github/main/profile/weightslab-banner-dark.png" alt="WeightsLab banner"></a>

  <a href="https://github.com/GrayboxTech/weightslab/blob/main/LICENSE"><img src="https://img.shields.io/badge/License-Apache%202.0-blue.svg" alt="License"></a>
  <a href="https://github.com/GrayboxTech/weightslab/stargazers"><img src="https://img.shields.io/github/stars/GrayboxTech/weightslab?style=flat&color=5865F2" alt="Stars"></a>
  <a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?style=flat&color=5865F2&logo=pypi&logoColor=white" alt="Version"></a>
  <br>
  <a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/main/weightslab/examples/Notebooks/PyTorch/ws-classification.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open WeightsLab In Colab"></a>

  Welcome to the WeightsLab image-classification notebook! <a href="https://github.com/GrayboxTech/weightslab">WeightsLab</a> is an open-source PyTorch tool for dataset debugging, mislabel detection, and mid-training data curation. Browse the <a href="https://grayboxtech.github.io/weightslab/latest/index.html">Docs</a> for details, and raise an issue on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a> for support.</div>

# Bank Fraud Detection with WeightsLab (tabular)

This notebook trains a small MLP to flag **fraudulent bank card transactions** and instruments it with WeightsLab so every training signal is traced **back to the exact transaction** producing it.

There are no images: a sample is one transaction (a **row**), the model input **is** the 16-feature vector, and WeightsLab sends that vector to the UI as a `vector` (not a fake image). Each raw feature (`amount`, `merchant_risk`, `geo_distance_km`, …) is a **sortable column** in the List Exploration view.

### What you'll do
1. Install WeightsLab.
2. Generate a reproducible synthetic fraud stream (no download).
3. Wrap the model, optimizer, dataloaders, loss and metric.
4. Train while streaming per-transaction loss/prediction to Weights Studio.

*Real drop-in datasets: Kaggle Credit Card Fraud (ULB), PaySim.*

## Setup

Install WeightsLab from PyPI. These tabular demos are tiny — the free Colab CPU runtime is plenty (no GPU needed).

> The **tabular input path** (the feature vector is sent to the UI as a `vector` — not a fake image) needs a WeightsLab build with tabular support (the `252-tabular-experiment` line or a release that includes it).

<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?color=5865F2&logo=pypi&logoColor=white" alt="PyPI - Version"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/dm/weightslab?color=5865F2" alt="PyPI - Downloads"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/pyversions/weightslab?color=5865F2&logo=python&logoColor=white" alt="PyPI - Python Version"></a>

In [ ]:
%pip install torch torchvision

In [ ]:
%pip install --pre --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "weightslab==1.3.3.dev6"

## 1. Imports

`weightslab` is imported as `wl`. The two `guard_*_context` managers scope a block as training vs. evaluation so signals are attributed to the right phase.

In [ ]:
import tempfile, logging
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torchmetrics.classification import Accuracy
from tqdm.auto import tqdm

import weightslab as wl
from weightslab.components.global_monitoring import (
    guard_training_context,
    guard_testing_context,
)

logging.basicConfig(level=logging.ERROR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. A synthetic fraud dataset (rows, not images)

`make_synthetic_fraud` builds reproducible transactions: 16 numeric features with a binary label (~12% fraud). Fraud rows come from shifted distributions (larger amounts, off-hours activity, higher merchant risk, device changes, larger geo jumps). The dataset returns the **standardized feature vector** as the model input, and `get_items(...)` exposes the **raw** values as metadata columns.

In [ ]:
FEATURE_NAMES = [
    "amount", "old_balance", "new_balance", "balance_delta",
    "txn_hour", "txn_day_of_week", "txn_count_1h", "txn_count_24h",
    "avg_amount_7d", "std_amount_7d", "merchant_risk", "device_change",
    "geo_distance_km", "is_foreign", "account_age_days", "num_prior_disputes",
]
NUM_FEATURES = len(FEATURE_NAMES)  # 16
FRAUD_RATE = 0.12


def make_synthetic_fraud(n_samples, seed=0):
    """Return (X_std, X_raw, y): standardized input, raw display values, label."""
    rng = np.random.default_rng(seed)
    n_fraud = max(1, round(n_samples * FRAUD_RATE))
    n_legit = max(1, n_samples - n_fraud)

    def draw(n, fraud):
        amount = rng.gamma(2.0, 180.0 if fraud else 60.0, n)
        old_balance = rng.gamma(2.0, 800.0, n)
        spent = amount * (rng.uniform(0.6, 1.4, n) if fraud else rng.uniform(0.0, 0.6, n))
        new_balance = np.clip(old_balance - spent, 0, None)
        balance_delta = old_balance - new_balance
        txn_hour = (rng.normal(2.5, 2.0, n) % 24) if fraud else rng.normal(13.0, 4.0, n)
        txn_dow = rng.integers(0, 7, n).astype(float)
        txn_count_1h = rng.poisson(4.0 if fraud else 1.0, n).astype(float)
        txn_count_24h = rng.poisson(18.0 if fraud else 6.0, n).astype(float)
        avg_amount_7d = rng.gamma(2.0, 90.0 if fraud else 70.0, n)
        std_amount_7d = rng.gamma(2.0, 60.0 if fraud else 25.0, n)
        merchant_risk = rng.beta(5.0, 2.0, n) if fraud else rng.beta(2.0, 6.0, n)
        device_change = rng.binomial(1, 0.55 if fraud else 0.08, n).astype(float)
        geo = rng.gamma(2.0, 400.0 if fraud else 25.0, n)
        is_foreign = rng.binomial(1, 0.45 if fraud else 0.05, n).astype(float)
        account_age = rng.gamma(2.0, 120.0 if fraud else 500.0, n)
        disputes = rng.poisson(1.5 if fraud else 0.2, n).astype(float)
        return np.stack([amount, old_balance, new_balance, balance_delta, txn_hour,
                         txn_dow, txn_count_1h, txn_count_24h, avg_amount_7d,
                         std_amount_7d, merchant_risk, device_change, geo,
                         is_foreign, account_age, disputes], axis=1)

    x_raw = np.concatenate([draw(n_legit, False), draw(n_fraud, True)]).astype(np.float64)
    y = np.concatenate([np.zeros(n_legit, np.int64), np.ones(n_fraud, np.int64)])
    mean, std = x_raw.mean(0, keepdims=True), x_raw.std(0, keepdims=True)
    std[std == 0] = 1.0
    x_std = (x_raw - mean) / std
    perm = rng.permutation(len(x_raw))
    return x_std[perm].astype(np.float32), x_raw[perm].astype(np.float32), y[perm]


class FraudDataset(Dataset):
    """Yields (feature_vector, id, label); get_items() adds feature columns."""

    def __init__(self, n_samples, seed=0):
        xs, xr, y = make_synthetic_fraud(n_samples, seed)
        self.features = torch.from_numpy(xs)   # model input
        self.raw = xr                          # display values
        self.labels = torch.from_numpy(y)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], idx, int(self.labels[idx])

    def get_items(self, idx, include_metadata=False, include_labels=False, include_images=False):
        x = self.features[idx] if include_images else None
        target = int(self.labels[idx]) if include_labels else None
        meta = ({n: round(float(self.raw[idx][i]), 4) for i, n in enumerate(FEATURE_NAMES)}
                if include_metadata else None)
        return x, idx, target, meta


class FraudMLP(nn.Module):
    def __init__(self, in_features=NUM_FEATURES, hidden=64, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(), nn.Linear(in_features, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, num_classes))

    def forward(self, x):
        return self.net(x)


def build_dataset(n, seed=0):
    return FraudDataset(n, seed)


def build_model():
    return FraudMLP()


## 3. Configuration

Every tunable lives here in one dict, like a `config.yaml` with comments. Wrapping it with `flag="hyperparameters"` lets Weights Studio read (and live-edit) these values while training.

In [ ]:
config = {
    "experiment_name": "fraud_detection_mlp",
    "device": str(device),
    "root_log_dir": tempfile.mkdtemp(prefix="weightslab_fraud_"),
    "learning_rate": 0.005,
    "training_steps_to_do": 2000,
    "eval_full_to_train_steps_ratio": 100,
    "write_export_ratio": 500,
    "class_weights": [1.0, 4.0],           # [legit, fraud] — up-weight fraud
    "dataset": {"seed": 0, "n_train": 4000, "n_test": 1000},
    "data": {"train_loader": {"batch_size": 32},
             "test_loader": {"batch_size": 128}},
}
wl.watch_or_edit(config, flag="hyperparameters", poll_interval=1.0)

## 4. Wrap the training objects

This is the heart of WeightsLab. Each object passes through `wl.watch_or_edit(...)` with a `flag` describing its role. The tracked dataset's `get_items()` exposes every feature as a **sortable column**; `preload_metadata=True` loads them at init.

In [ ]:
cfg = config
train_ds = build_dataset(cfg['dataset']['n_train'], seed=cfg['dataset']['seed'])
test_ds  = build_dataset(cfg['dataset']['n_test'],  seed=cfg['dataset']['seed'] + 1)

model = wl.watch_or_edit(build_model().to(device), flag='model', device=device)
optimizer = wl.watch_or_edit(
    optim.Adam(model.parameters(), lr=cfg['learning_rate']), flag='optimizer')

train_loader = wl.watch_or_edit(
    train_ds, flag='data', loader_name='train_loader',
    batch_size=cfg['data']['train_loader']['batch_size'], shuffle=True,
    is_training=True, preload_labels=True, preload_metadata=True)
test_loader = wl.watch_or_edit(
    test_ds, flag='data', loader_name='test_loader',
    batch_size=cfg['data']['test_loader']['batch_size'], shuffle=False,
    is_training=False, preload_labels=True, preload_metadata=True)

# Class weights counter the minority (positive) class prevalence.
cw = torch.tensor(cfg['class_weights'], dtype=torch.float32, device=device)
train_criterion = wl.watch_or_edit(
    nn.CrossEntropyLoss(weight=cw, reduction='none'),
    flag='loss', signal_name='train-loss-CE', log=True)
test_criterion = wl.watch_or_edit(
    nn.CrossEntropyLoss(weight=cw, reduction='none'),
    flag='loss', signal_name='test-loss-CE', log=True)
metric = wl.watch_or_edit(
    Accuracy(task='multiclass', num_classes=2).to(device),
    flag='metric', signal_name='metric-ACC', log=True)

## 5. Train and evaluate steps

The `guard_training_context` / `guard_testing_context` blocks tell WeightsLab the phase. `criterion(..., batch_ids=ids, preds=preds)` passes the sample ids so loss is stored **per sample**, and `wl.save_signals(...)` logs a custom per-sample accuracy signal during evaluation.

In [ ]:
def train(loader, model, optimizer, criterion, device):
    with guard_training_context:
        inputs, ids, labels = next(loader)
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(inputs)
        preds = logits.argmax(dim=1, keepdim=True)
        loss = criterion(logits.float(), labels.long(), batch_ids=ids, preds=preds)
        total = loss.mean()
        total.backward()
        optimizer.step()
    return total.detach().cpu().item()


def test(loader, model, criterion, metric, device, n_batches):
    losses = torch.tensor(0.0, device=device)
    for inputs, ids, labels in loader:
        with guard_testing_context:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = model(inputs)
            preds = logits.argmax(dim=1, keepdim=True)
            losses += criterion(logits, labels, batch_ids=ids, preds=preds).mean()
            metric.update(logits, labels)
            correct = (preds.view(-1) == labels.view(-1)).float()
            wl.save_signals(preds_raw=logits, targets=labels, batch_ids=ids,
                            signals={"test_metric/Accuracy_per_sample": correct}, preds=preds)
    return (losses / max(1, n_batches)).item(), (metric.compute() * 100).item()


## 6. Serve and train

`wl.serve(serving_grpc=True, serving_bore=True)` starts the background gRPC server (non-blocking) and a `bore.pub` tunnel so Weights Studio on your own machine can reach this Colab backend. `wl.start_training(...)` flips the experiment into the *training* state, then we run the loop.

In [ ]:
wl.serve(serving_grpc=True, serving_bore=True)

In [ ]:
wl.start_training(timeout=3)

steps = config['training_steps_to_do']
eval_ratio = config['eval_full_to_train_steps_ratio']
n_test_batches = len(test_loader)

test_loss = test_acc = None
pbar = tqdm(range(steps), desc='Training')
for step in pbar:
    age = model.get_age() if hasattr(model, 'get_age') else step
    train_loss = train(train_loader, model, optimizer, train_criterion, device)
    if age > 0 and age % eval_ratio == 0:
        test_loss, test_acc = test(test_loader, model, test_criterion, metric, device, n_test_batches)
    postfix = {'loss': f'{train_loss:.3f}'}
    if test_acc is not None:
        postfix['test_acc'] = f'{test_acc:.1f}%'
    pbar.set_postfix(postfix)

wl.write_history()
wl.write_dataframe()
print('Training complete.')

## See it live in Weights Studio

Everything above runs headless. The payoff is **Weights Studio**, where each row is one record and every feature is a **sortable column**.

Studio runs as a local Docker stack, and **Colab has no Docker daemon**, so you run Studio on your own machine and point it at this notebook's backend via the `bore.pub:<port>` endpoint **printed in Section 6**.

**On your machine** (with Docker Desktop):
```bash
pip install weightslab
weightslab ui launch                 # opens http://localhost:5173
weightslab tunnel bore.pub:12345     # the host:port printed in Section 6
```

Then open **http://localhost:5173** and switch the Data Exploration board to the **List** view.

## Curate in the UI

In Weights Studio, switch the Data Exploration board to the **List** view:

1. **Sort by `train-loss-CE` descending** and generate its histogram to find the hardest transactions — usually the frauds the model still misses.
2. **Lock** the loss sort, then add `merchant_risk` or `amount` as a secondary sort to see which feature ranges drive the errors.
3. **Right-click a column** to clone it, reset the sort, or generate a histogram — the same actions you use on image datasets.
4. **Discard** mislabeled or leaked rows and keep training — no restart.

---

<div align="center">
Crafted by <a href="https://github.com/GrayboxTech/weightslab">GrayboxTech</a> - if WeightsLab helps you catch a bad label, drop us a star on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a>.
</div>